# 02 — Sentiment Analysis
Thử nghiệm và đánh giá các mô hình phân tích cảm xúc tiếng Việt.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.nlp.sentiment_analyzer import SentimentAnalyzer

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Khởi tạo analyzer (lần đầu sẽ tải model ~1.1GB)
print("⏳ Đang khởi tạo SentimentAnalyzer...")
analyzer = SentimentAnalyzer()
print("✅ Model đã sẵn sàng!")

## 1. Test nhanh với các câu ví dụ

In [ ]:
# Test với các câu tiếng Việt
test_texts = [
    "Video này hay quá, cảm ơn bạn đã chia sẻ!",
    "Thật tệ hại, không như mong đợi chút nào.",
    "Hôm nay trời đẹp, tôi đi dạo công viên.",
    "Dịch vụ này quá tệ, chậm chạp và đắt đỏ.",
    "Tôi thích sản phẩm này, chất lượng rất tốt!",
    "Bình thường thôi, không có gì đặc biệt.",
]

print("=== TEST DỰ ĐOÁN CẢM XÚC ===\n")
results = []
for text in test_texts:
    result = analyzer.predict(text)
    results.append({
        'text': text,
        'sentiment': result['label'],
        'confidence': result['score']
    })
    print(f"📝 {text[:50]}...")
    print(f"   → {result['label'].upper()} ({result['score']*100:.1f}%)\n")

results_df = pd.DataFrame(results)
results_df

## 2. Phân tích trên dataset thực tế

In [ ]:
# Load dữ liệu YouTube comments
DATA_PATH = Path('../data/processed/youtube_comments.csv')

if DATA_PATH.exists():
    print(f"📊 Đang đọc dữ liệu từ {DATA_PATH}...")
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Đã load {len(df)} bình luận")
    print(f"\nColumns: {list(df.columns)}")
    df.head()
else:
    print(f"❌ Không tìm thấy file: {DATA_PATH}")
    print("Hãy chạy youtube_crawler trước!")
    df = None

In [ ]:
# Phân tích cảm xúc toàn bộ dataset
if df is not None and 'text' in df.columns:
    print(f"\n⏳ Bắt đầu phân tích {len(df)} bình luận...")
    print("(Có thể mất vài phút với dataset lớn)\n")
    
    # Batch predict
    texts = df['text'].fillna("").tolist()
    results = analyzer.batch_predict(texts)
    
    # Thêm vào DataFrame
    df['sentiment'] = [r['label'] for r in results]
    df['confidence'] = [r['score'] for r in results]
    
    print("\n=== KẾT QUẢ PHÂN TÍCH ===")
    print(df['sentiment'].value_counts())
    
    # Lưu kết quả
    output_path = DATA_PATH.parent / 'youtube_comments_sentiment.csv'
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n💾 Đã lưu vào: {output_path}")
    
    df.head()
else:
    print("❌ Không có dữ liệu để phân tích")

## 3. Trực quan hóa kết quả

In [ ]:
if df is not None and 'sentiment' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # 1. Count plot
    sentiment_counts = df['sentiment'].value_counts()
    colors = ['#22c55e', '#94a3b8', '#ef4444']
    axes[0, 0].bar(sentiment_counts.index, sentiment_counts.values, color=colors)
    axes[0, 0].set_title('Phân bố cảm xúc', fontsize=14, fontweight='bold')
    axes[0, 0].set_ylabel('Số lượng')
    for i, v in enumerate(sentiment_counts.values):
        axes[0, 0].text(i, v + 5, str(v), ha='center', fontweight='bold')
    
    # 2. Pie chart
    axes[0, 1].pie(sentiment_counts.values, labels=sentiment_counts.index, 
                   autopct='%1.1f%%', colors=colors, startangle=90)
    axes[0, 1].set_title('Tỷ lệ cảm xúc', fontsize=14, fontweight='bold')
    
    # 3. Confidence distribution
    if 'confidence' in df.columns:
        axes[1, 0].hist(df[df['sentiment'] == 'positive']['confidence'], alpha=0.6, label='Positive', color='#22c55e')
        axes[1, 0].hist(df[df['sentiment'] == 'neutral']['confidence'], alpha=0.6, label='Neutral', color='#94a3b8')
        axes[1, 0].hist(df[df['sentiment'] == 'negative']['confidence'], alpha=0.6, label='Negative', color='#ef4444')
        axes[1, 0].set_title('Phân bố độ tin cậy theo cảm xúc', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Confidence')
        axes[1, 0].set_ylabel('Số lượng')
        axes[1, 0].legend()
    
    # 4. Average confidence by sentiment
    if 'confidence' in df.columns:
        avg_conf = df.groupby('sentiment')['confidence'].mean()
        axes[1, 1].bar(avg_conf.index, avg_conf.values, color=colors)
        axes[1, 1].set_title('Độ tin cậy TB theo cảm xúc', fontsize=14, fontweight='bold')
        axes[1, 1].set_ylabel('Confidence')
        for i, v in enumerate(avg_conf.values):
            axes[1, 1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Thống kê chi tiết
    print("\n=== THỐNG KÊ CHI TIẾT ===")
    print(f"\nTổng số bình luận: {len(df):,}")
    print(f"\nPhân bố cảm xúc:")
    for sentiment in ['positive', 'neutral', 'negative']:
        count = (df['sentiment'] == sentiment).sum()
        pct = count / len(df) * 100
        avg_conf = df[df['sentiment'] == sentiment]['confidence'].mean() if 'confidence' in df.columns else 0
        print(f"  {sentiment.upper()}: {count:,} ({pct:.1f}%) - Confidence TB: {avg_conf:.2%}")
else:
    print("❌ Không có dữ liệu để hiển thị")

## 4. Phân tích chi tiết

In [ ]:
if df is not None and 'sentiment' in df.columns:
    # Top bình luận tiêu cực có confidence cao
    print("=== TOP 10 BÌNH LUẬN TIÊU CỰC ĐÁNG CHÚ Ý ===\n")
    negative_comments = df[df['sentiment'] == 'negative'].sort_values('confidence', ascending=False)
    
    for idx, row in negative_comments.head(10).iterrows():
        text = row.get('text', row.get('description', ''))[:80]
        conf = row['confidence']
        print(f"⚠️  [{conf:.1%}] {text}...")
    
    # Top bình luận tích cực
    print("\n=== TOP 10 BÌNH LUẬN TÍCH CỰC ===\n")
    positive_comments = df[df['sentiment'] == 'positive'].sort_values('confidence', ascending=False)
    
    for idx, row in positive_comments.head(10).iterrows():
        text = row.get('text', row.get('description', ''))[:80]
        conf = row['confidence']
        print(f"✅ [{conf:.1%}] {text}...")
else:
    print("❌ Không có dữ liệu")

## 5. So sánh với các mô hình khác (Placeholder)

In [ ]:
# TODO: Implement so sánh với các mô hình khác
# - TF-IDF + Logistic Regression
# - SVM
# - ViSoBERT (PhoBERT fine-tuned)
# - VADER (cho tiếng Việt)
#
# Metrics để so sánh:
# - Accuracy
# - F1-score
# - Precision & Recall
# - Inference time
# - Model size

print("📌 Phần này sẽ được implement sau để so sánh các mô hình sentiment analysis khác nhau.")
print("\nCác mô hình cần so sánh:")
print("  1. XLM-RoBERTa (hiện tại) - Multilingual, ổn định")
print("  2. ViSoBERT - Fine-tuned cho tiếng Việt")
print("  3. TF-IDF + Logistic Regression - Traditional ML")
print("  4. SVM - Traditional ML")

## 6. Kết luận

In [ ]:
if df is not None and 'sentiment' in df.columns:
    print("=== KẾT LUẬN ===\n")
    
    neg_ratio = (df['sentiment'] == 'negative').mean()
    pos_ratio = (df['sentiment'] == 'positive').mean()
    avg_conf = df['confidence'].mean()
    
    print(f"✅ Đã phân tích {len(df)} bình luận")
    print(f"\n📊 Tỷ lệ cảm xúc:")
    print(f"   - Tích cực: {pos_ratio:.1%}")
    print(f"   - Tiêu cực: {neg_ratio:.1%}")
    print(f"   - Trung tính: {1 - pos_ratio - neg_ratio:.1%}")
    print(f"\n🎯 Độ tin cậy TB: {avg_conf:.1%}")
    
    if neg_ratio > 0.4:
        print("\n⚠️  CẢNH BÁO: Tỷ lệ tiêu cực cao, cần theo dõi!")
    else:
        print("\n✅ Tâm lý thảo luận ở mức an toàn.")
    
    print("\n✅ Sentiment analysis hoàn thành!")
else:
    print("❌ Chưa có dữ liệu để đánh giá")